In [ ]:
import dlt
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, TimestampType

In [ ]:
# In DLT pipelines, spark configuration (like "catalog") is set at the pipeline level
# (in the DLT pipeline's settings/YAML), read here via spark.conf instead of sys.argv
# or dbutils.widgets — DLT notebooks use their own parameter-passing mechanism
catalog = spark.conf.get("catalog")

In [ ]:
# Explicitly define the schema for the Bronze layer, converting raw string fields
# from Landing (started_at/ended_at, lat/lng) into proper timestamp and decimal types
schema = StructType([
    StructField("ride_id", StringType(), True),
    StructField("rideable_type", StringType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("ended_at", TimestampType(), True),
    StructField("start_station_name", StringType(), True),
    StructField("start_station_id", StringType(), True),
    StructField("end_station_name", StringType(), True),
    StructField("end_station_id", StringType(), True),
    StructField("start_lat", DecimalType(9, 6), True),
    StructField("start_lng", DecimalType(9, 6), True),
    StructField("end_lat", DecimalType(9, 6), True),
    StructField("end_lng", DecimalType(9, 6), True),
    StructField("member_casual", StringType(), True)
])

In [ ]:
# @dlt.table turns this function into a declarative table definition — instead of
# manually writing df.write.saveAsTable(...) as before, DLT infers the table name
# from the function name (bronze_jc_citibike) and manages the write, scheduling,
# and dependency tracking automatically based on how this function is used elsewhere
@dlt.table(
    comment="Bronze layer: Raw Citibike with ingest metadata"
)
def bronze_jc_citibike():
    df = (
        spark.read.schema(schema).csv(f"/Volumes/{catalog}/00_landing/source_citibike_data/JC-202503-citibike-tripdata.csv", header=True)
    )
    return df